In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import recall_score, precision_score
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Load data
train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test.csv')

coil_ids = test['CoilID'].values
X_train = train.drop(['CoilID', 'Y'], axis=1).copy()
y_train = train['Y'].astype(int)
X_test  = test.drop(['CoilID'], axis=1).copy()

# Fill missing values
for col in X_train.columns:
    med = X_train[col].median()
    X_train[col] = X_train[col].fillna(med)
    X_test[col]  = X_test[col].fillna(med)

# Feature engineering
for df in [X_train, X_test]:
    v = df.values.copy()
    df['mean_all']   = v.mean(axis=1)
    df['std_all']    = v.std(axis=1)
    df['max_all']    = v.max(axis=1)
    df['min_all']    = v.min(axis=1)
    df['range_all']  = df['max_all'] - df['min_all']
    df['iqr_all']    = np.quantile(v,0.75,axis=1) - np.quantile(v,0.25,axis=1)
    z = (v - v.mean(axis=1,keepdims=True)) / (v.std(axis=1,keepdims=True)+1e-9)
    df['n_outliers'] = (np.abs(z)>2).sum(axis=1)
    df['cv']         = df['std_all'] / (np.abs(df['mean_all'])+1e-9)

# Scale
scaler = RobustScaler()
X_tr = scaler.fit_transform(X_train)
X_te = scaler.transform(X_test)

# SMOTE
sm = SMOTE(random_state=42, sampling_strategy=0.5)
X_res, y_res = sm.fit_resample(X_tr, y_train)

# Train XGBoost
model = xgb.XGBClassifier(
    n_estimators=800, max_depth=4, learning_rate=0.01,
    scale_pos_weight=10, subsample=0.8, colsample_bytree=0.7,
    min_child_weight=3, gamma=1, eval_metric='logloss', random_state=42
)
model.fit(X_res, y_res)

# Threshold = 0.870
THRESHOLD = 0.870
train_proba = model.predict_proba(X_tr)[:,1]
train_preds = (train_proba >= THRESHOLD).astype(int)
print(f"Recall    : {recall_score(y_train, train_preds):.4f}")
print(f"Precision : {precision_score(y_train, train_preds):.4f}")

# Predict & save
test_proba  = model.predict_proba(X_te)[:,1]
final_preds = (test_proba >= THRESHOLD).astype(int)

submission = pd.DataFrame({'CoilID': coil_ids, 'Y': final_preds})
submission.to_csv('expected_submission.csv', index=False)
print(f"Defects detected: {sum(final_preds)} / {len(final_preds)}")
print("Done!")

Recall    : 1.0000
Precision : 0.9565
Defects detected: 5 / 339
Done!
